# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [14]:
# Load the dataset
df = pd.read_csv("data/AviationData.csv", encoding="latin1")

# Preview the dataset
display(df.head())

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Inspect datatypes and non-null values
df.info()

# Inspect missing values
display(df.isna().sum())

# Print summary statistics
display(df.describe())

C:\Users\mnjihia\AppData\Local\Temp\ipykernel_39392\1079593835.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="latin1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


Dataset shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make       

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [15]:
# Inspect the relevant columns
print("Aircraft Categories:")
print(df["Aircraft.Category"].value_counts(dropna=False))

print("\nAmateur Built:")
print(df["Amateur.Built"].value_counts(dropna=False))

print("\nMissing Values:")
print(df[["Aircraft.Category", "Amateur.Built", "Event.Date"]].isna().sum())

# Convert Event.Date to datetime
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

# Remove rows with missing values in the columns needed for filtering
df = df.dropna(subset=["Aircraft.Category", "Amateur.Built", "Event.Date"])

# Keep only airplanes
df = df[df["Aircraft.Category"] == "Airplane"]

# Keep only professionally built aircraft
df = df[df["Amateur.Built"] == "No"]

# Keep accidents from 1983 onwards
df = df[df["Event.Date"].dt.year >= 1983]

# Check the filtered dataset
print("\nFiltered Dataset Shape:")
print(df.shape)

print("\nAircraft Category:")
print(df["Aircraft.Category"].value_counts())

print("\nAmateur Built:")
print(df["Amateur.Built"].value_counts())

print("\nEarliest Event Date:")
print(df["Event.Date"].min())

Aircraft Categories:
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

Amateur Built:
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

Missing Values:
Aircraft.Category    56602
Amateur.Built          102
Event.Date               0
dtype: int64

Filtered Dataset Shape:
(21447, 31)

Aircraft Category:
Aircraft.Category
Airplane    21447
Name: count, dtype: int64

Amateur Built:
Amateur.Built
No    21447
Name: count, dtype: int64

Earliest Event Date:
1983-03-18 00:00:00


Missing values in the injury columns were treated as zero. Total passengers were estimated by adding fatal, serious, minor and uninjured persons. The serious/fatal injury rate was calculated by dividing fatal and serious injuries by the estimated total number of passengers. Records with zero total passengers were excluded because an injury rate could not be calculated.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [16]:
# Fill missing injury values with 0
df["Total.Fatal.Injuries"] = df["Total.Fatal.Injuries"].fillna(0)
df["Total.Serious.Injuries"] = df["Total.Serious.Injuries"].fillna(0)
df["Total.Minor.Injuries"] = df["Total.Minor.Injuries"].fillna(0)
df["Total.Uninjured"] = df["Total.Uninjured"].fillna(0)

# Estimate total number of passengers
df["Total.Passengers"] = (
    df["Total.Fatal.Injuries"]
    + df["Total.Serious.Injuries"]
    + df["Total.Minor.Injuries"]
    + df["Total.Uninjured"]
)

# Calculate total serious and fatal injuries
df["Serious.Fatal.Injuries"] = (
    df["Total.Fatal.Injuries"]
    + df["Total.Serious.Injuries"]
)

# Remove records where no passengers were recorded
df = df[df["Total.Passengers"] > 0]

# Calculate serious/fatal injury rate
df["Serious.Fatal.Injury.Rate"] = (
    df["Serious.Fatal.Injuries"] / df["Total.Passengers"]
)

# View the first 5 results
df[[
    "Total.Passengers",
    "Serious.Fatal.Injuries",
    "Serious.Fatal.Injury.Rate"
]].head()

,Total.Passengers,Serious.Fatal.Injuries,Serious.Fatal.Injury.Rate
4149,588.0,0.0,0.0
4150,588.0,0.0,0.0
4171,2.0,2.0,1.0
4285,5.0,1.0,0.2
5957,289.0,0.0,0.0


Missing values in the injury columns were treated as zero. Total passengers were estimated by adding fatal, serious, minor, and uninjured persons. The serious/fatal injury rate was calculated by dividing fatal and serious injuries by the estimated total number of passengers. Records with zero total passengers were excluded because an injury rate could not be calculated.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [17]:
# Check the values in Aircraft.damage
print(df["Aircraft.damage"].value_counts(dropna=False))

# Remove rows where Aircraft.damage is missing
df = df.dropna(subset=["Aircraft.damage"])

# Create a new column: 1 = Destroyed, 0 = Not Destroyed
df["Aircraft.Destroyed"] = (
    df["Aircraft.damage"] == "Destroyed"
).astype(int)

# Check the result
print(df["Aircraft.Destroyed"].value_counts())

Aircraft.damage
Substantial    16821
Destroyed       2268
NaN              787
Minor            593
Unknown           74
Name: count, dtype: int64
Aircraft.Destroyed
0    17488
1     2268
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [18]:
# Check the Make column
print(df["Make"].isna().sum())
print(df["Make"].value_counts().head(10))

# Remove missing Make values
df = df.dropna(subset=["Make"])

# Clean the Make names
df["Make"] = df["Make"].str.strip().str.upper()

# Count the number of records for each Make
make_counts = df["Make"].value_counts()

# Keep Makes with at least 50 records
valid_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(valid_makes)]

# Check the cleaned Make column
print("Number of Makes remaining:", df["Make"].nunique())
print(df["Make"].value_counts().head(10))

1
Make
CESSNA                4724
PIPER                 2753
Cessna                2252
Piper                 1177
BEECH                  990
Beech                  405
BOEING                 368
MOONEY                 233
AIR TRACTOR INC        217
CIRRUS DESIGN CORP     215
Name: count, dtype: int64
Number of Makes remaining: 34
Make
CESSNA                6976
PIPER                 3930
BEECH                 1395
BOEING                 486
MOONEY                 357
AIR TRACTOR INC        219
BELLANCA               218
CIRRUS DESIGN CORP     217
MAULE                  215
AIR TRACTOR            203
Name: count, dtype: int64


Missing values in the Make column were removed. Manufacturer names were standardized by removing extra spaces and converting them to uppercase so that different versions of the same manufacturer were treated as one group. Makes with fewer than 50 accident records were excluded to ensure that later comparisons are based on a reasonable number of observations.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [19]:
# Remove rows with missing Make or Model
df = df.dropna(subset=["Make", "Model"])

# Clean the Make and Model names
df["Make"] = df["Make"].str.strip().str.upper()
df["Model"] = df["Model"].str.strip().str.upper()

# Create a unique identifier using Make and Model
df["Make_Model"] = df["Make"] + " " + df["Model"]

# Inspect the number of records for each make/model
print(df["Make_Model"].value_counts().head(20))

# Check the result
df[["Make", "Model", "Make_Model"]].head()

Make_Model
CESSNA 172                 745
CESSNA 152                 308
CESSNA 182                 299
CESSNA 172S                267
PIPER PA28                 263
CESSNA 172N                244
CESSNA 180                 213
CESSNA 172M                178
PIPER PA-18-150            175
CESSNA 150                 172
PIPER PA-28-140            166
BEECH A36                  162
CIRRUS DESIGN CORP SR22    142
CESSNA 172P                141
CESSNA 140                 115
BOEING 737                 113
CESSNA 170B                107
CESSNA 172R                106
PIPER PA-28-180            105
PIPER PA-28-161            102
Name: count, dtype: int64


,Make,Model,Make_Model
4150,BOEING,747,BOEING 747
4171,PIPER,PA-28-140,PIPER PA-28-140
6760,BOEING,727-200,BOEING 727-200
6806,BEECH,C35,BEECH C35
7084,CESSNA,180K,CESSNA 180K


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [20]:
# Inspect other columns that may affect accident outcomes

columns_to_check = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

# Check missing values
print("Missing Values:")
print(df[columns_to_check].isna().sum())

# Check values in each column
for column in columns_to_check:
    print("\n", column)
    print(df[column].value_counts(dropna=False))

Missing Values:
Engine.Type               2303
Weather.Condition         1411
Number.of.Engines         1311
Purpose.of.flight         1716
Broad.phase.of.flight    13951
dtype: int64

 Engine.Type
Engine.Type
Reciprocating      12749
NaN                 2303
Turbo Prop           882
Turbo Fan            321
Unknown               64
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

 Weather.Condition
Weather.Condition
VMC    13945
NaN     1411
IMC      844
Unk      129
UNK       41
Name: count, dtype: int64

 Number.of.Engines
Number.of.Engines
1.0    13114
2.0     1890
NaN     1311
4.0       36
3.0       16
0.0        3
Name: count, dtype: int64

 Purpose.of.flight
Purpose.of.flight
Personal                     9791
Instructional                2386
NaN                          1716
Aerial Application            722
Business                      402
Positioning                   255
Unknown                   

In [21]:
# Clean Engine.Type
df["Engine.Type"] = df["Engine.Type"].replace({
    "UNK": "Unknown"
})

# Clean Weather.Condition
df["Weather.Condition"] = df["Weather.Condition"].replace({
    "Unk": "Unknown",
    "UNK": "Unknown"
})

# Replace 0 engines with a missing value
df["Number.of.Engines"] = df["Number.of.Engines"].replace(0, np.nan)

# Check the cleaned values
print("Engine Type:")
print(df["Engine.Type"].value_counts(dropna=False))

print("\nWeather Condition:")
print(df["Weather.Condition"].value_counts(dropna=False))

print("\nNumber of Engines:")
print(df["Number.of.Engines"].value_counts(dropna=False).sort_index())

Engine Type:
Engine.Type
Reciprocating      12749
NaN                 2303
Turbo Prop           882
Turbo Fan            321
Unknown               65
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
Name: count, dtype: int64

Weather Condition:
Weather.Condition
VMC        13945
NaN         1411
IMC          844
Unknown      170
Name: count, dtype: int64

Number of Engines:
Number.of.Engines
1.0    13114
2.0     1890
3.0       16
4.0       36
NaN     1314
Name: count, dtype: int64


Inconsistent unknown values in Engine.Type and Weather.Condition were standardized to Unknown. Records showing zero engines were treated as missing values because they are not meaningful for the airplane analysis. Other missing values were retained because there was not enough information to reliably replace them.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [22]:
# Inspect missing values in the current dataframe
print("Non-null values:")
print(df.notna().sum().sort_values())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nCurrent dataset shape:")
print(df.shape)

Non-null values:
Schedule                      1287
Broad.phase.of.flight         2419
Air.carrier                   7497
Airport.Code                 11154
Airport.Name                 11252
Report.Status                13568
Engine.Type                  14067
Purpose.of.flight            14654
Weather.Condition            14959
Number.of.Engines            15056
Longitude                    15356
Latitude                     15360
Publication.Date             15850
FAR.Description              16213
Registration.Number          16253
Location                     16367
Country                      16369
Model                        16370
Aircraft.Category            16370
Aircraft.damage              16370
Injury.Severity              16370
Accident.Number              16370
Investigation.Type           16370
Event.Id                     16370
Event.Date                   16370
Make                         16370
Total.Fatal.Injuries         16370
Amateur.Built                16370
Tot

The dataframe was inspected for missing values. At this stage, the dataset contained 16,370 records, so the stated threshold of more than 20,000 non-null values could not be applied because no column could meet that threshold. Therefore, no columns were removed using this threshold to avoid losing all variables needed for the analysis.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [23]:
# Save the cleaned dataframe to a CSV file
df.to_csv("data/cleaned_aviation_data.csv", index=False)

# Confirm that the file was saved
print("Cleaned data saved successfully.")
print("Final dataset shape:", df.shape)

Cleaned data saved successfully.
Final dataset shape: (16370, 36)
